In [8]:
import pandas as pd
import numpy as np
from pandas_ods_reader import read_ods
import matplotlib.pyplot as plt
from pyexcel import get_book

In [17]:
from odf.opendocument import load
from odf.table import Table

def get_ods_sheet_names(ods_file):
    doc = load(ods_file)  
    sheets = doc.spreadsheet.getElementsByType(Table)  
    sheet_names = [sheet.getAttribute("name") for sheet in sheets] 
    
    return sheet_names

ods_file = "Public_Testwork_Database.ods"
sheet_names = get_ods_sheet_names(ods_file)
data = [read_ods(ods_file, sheet=i) for i in range(len(sheet_names))]

In [18]:
print(sheet_names)

['Intro', 'Summary', 'LITHO', 'AI', 'DWT', 'PLI', 'SGI', 'UCS', 'WiBM', 'WiC', 'WiRM']


In [22]:
import psycopg2
from sqlalchemy import create_engine

DATABASE = 'postgres'
USER = 'user'
PASSWORD = '123'
HOST = 'localhost'
PORT = '5444'

connection_string = f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}'
engine = create_engine(connection_string)
for sheet_name, df in zip(sheet_names, data):
    df.to_sql(sheet_name, engine, if_exists='replace', index=False)

summary_df = data[2]

for i in range(3, len(sheet_names)):
    summary_df.merge(data[i], on='Id', how='outer')